# Comparing hedge policies across sensitivities

## Defining the benchmark hedges

At a preterminal date with remaining time $\tau$, the zero-rate Black-Scholes call delta is

$$
\Delta(S,\tau)=\Phi\!\left(\frac{\log(S/K)+\tfrac12\sigma^2\tau}{\sigma\sqrt{\tau}}\right).
$$

Here, $S$ is the current price, $K$ is the strike, $\sigma$ is the declared annual volatility, $\tau$ is remaining time, and $\Phi$ is the standard normal distribution function. This frictionless formula supplies a classical comparator. It is not treated as the optimum when costs are present.

### Common Independent Evaluation Paths

In [1]:
from pathlib import Path
import hashlib
import sys
import time

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from IPython.display import HTML, display as _display


def display(value):
    module = value.__class__.__module__
    if module.startswith("pandas") and hasattr(value, "_repr_html_"):
        html = value._repr_html_()
        return _display(HTML('<div style="display:flex;justify-content:center;">' + html + "</div>"))
    return _display(value)


BOOK_ROOT = Path.cwd().parents[1]
if str(BOOK_ROOT) not in sys.path:
    sys.path.insert(0, str(BOOK_ROOT))
from book_support import ensure_dataset
FINANCIAL_ROOT = BOOK_ROOT

import RiskLabAI
from RiskLabAI.backtest.deep_hedging import DeepHedge, black_scholes_call_delta, entropic_risk, gbm_hedging_paths, hedging_pnl, train_deep_hedge

assert RiskLabAI.__version__ == "3.2.0"

torch.set_num_threads(1)
torch.set_default_dtype(torch.float64)

FIGURES = BOOK_ROOT / "outputs" / "figures" / "31_deep_hedging"
TABLES = BOOK_ROOT / "outputs" / "tables" / "31_deep_hedging"
FIGURE_DATA = BOOK_ROOT / "outputs" / "figure_data" / "31_deep_hedging"
ARTIFACTS = BOOK_ROOT / "outputs" / "artifacts" / "31_deep_hedging"
DATA_FILE = ensure_dataset(BOOK_ROOT, "fred_nasdaq")
for directory in [FIGURES, TABLES, FIGURE_DATA, ARTIFACTS]:
    directory.mkdir(parents=True, exist_ok=True)

PALETTE = {
    "background": "#FFFFFF",
    "axes": "#FFFFFF",
    "text": "#1B365D",
    "cyan": "#0072B2",
    "amber": "#C76D00",
    "green": "#00866A",
    "rose": "#B83B3B",
    "violet": "#A03A78",
    "grid": "#D9DEE5",
}
plt.rcParams.update({
    "font.family": "DejaVu Serif",
    "font.size": 11.5,
    "figure.facecolor": PALETTE["background"],
    "axes.facecolor": PALETTE["axes"],
    "axes.edgecolor": PALETTE["text"],
    "axes.labelcolor": PALETTE["text"],
    "axes.titlecolor": PALETTE["text"],
    "xtick.color": PALETTE["text"],
    "ytick.color": PALETTE["text"],
    "text.color": PALETTE["text"],
    "grid.color": PALETTE["grid"],
    "grid.linestyle": ":",
    "legend.facecolor": PALETTE["axes"],
    "legend.edgecolor": PALETTE["grid"],
    "savefig.facecolor": PALETTE["background"],
    "savefig.transparent": False,
    "savefig.dpi": 300,
})


def rendered_value(value):
    if pd.isna(value):
        return ""
    if isinstance(value, (bool, np.bool_)):
        return "Pass" if bool(value) else "Fail"
    if isinstance(value, (int, np.integer)):
        return f"{int(value):,}"
    if isinstance(value, (float, np.floating)):
        value = float(value)
        if value == 0:
            return "0"
        if value.is_integer():
            return f"{int(value):,}"
        if abs(value) < 0.0001 or abs(value) >= 1000000:
            return f"{value:.6g}"
        return f"{value:.6f}".rstrip("0").rstrip(".")
    return str(value)


def rendered_frame(frame):
    return frame.apply(lambda column: column.map(rendered_value))


def save_table(frame, stem, caption, label, fit=False):
    shown = rendered_frame(frame)
    shown.to_csv(TABLES / f"{stem}.csv", index=False)
    latex = shown.to_latex(index=False, escape=True, caption=caption, label=label, position="htbp")
    if fit:
        latex = latex.replace("\\begin{tabular}", "\\begin{adjustbox}{max width=\\textwidth}\\begin{tabular}", 1).replace("\\end{tabular}", "\\end{tabular}\\end{adjustbox}", 1)
    (TABLES / f"{stem}.tex").write_text(latex, encoding="utf-8")
    display(shown)
    return shown


def save_split_table(frame, groups, stem, caption, label):
    shown = rendered_frame(frame)
    shown.to_csv(TABLES / f"{stem}.csv", index=False)
    panels = []
    for heading, columns in groups:
        tabular = shown[columns].to_latex(index=False, escape=True)
        tabular = tabular.replace("\\begin{tabular}", "\\begin{adjustbox}{max width=\\textwidth}\\begin{tabular}", 1).replace("\\end{tabular}", "\\end{tabular}\\end{adjustbox}", 1)
        panels.append(f"\\textit{{{heading}}}\\par\\smallskip\n{tabular}")
    latex = f"\\begin{{table}}[htbp]\n\\caption{{{caption}}}\n\\label{{{label}}}\n\\centering\n\\small\n" + "\\medskip\n".join(panels) + "\\end{table}\n"
    (TABLES / f"{stem}.tex").write_text(latex, encoding="utf-8")
    display(shown)
    return shown


def finish_axis(axis, grid=True):
    axis.spines["top"].set_visible(False)
    axis.spines["right"].set_visible(False)
    if grid:
        axis.grid(True, alpha=0.65)
    axis.set_axisbelow(True)


def save_figure(figure, stem):
    figure.savefig(FIGURES / f"{stem}.png", dpi=300, bbox_inches="tight", facecolor=PALETTE["background"], transparent=False)
    figure.savefig(FIGURES / f"{stem}.pdf", bbox_inches="tight", facecolor=PALETTE["background"], transparent=False)
    display(figure)
    plt.close(figure)


def file_sha256(path):
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest().upper()


def public_volatility():
    frame = pd.read_csv(DATA_FILE, parse_dates=["observation_date"])
    frame["NASDAQCOM"] = pd.to_numeric(frame["NASDAQCOM"], errors="coerce")
    design = frame.loc[(frame["observation_date"] >= "1994-01-04") & (frame["observation_date"] <= "2014-12-31")].copy()
    observed = design.dropna(subset=["NASDAQCOM"]).copy()
    observed["Log return"] = np.log(observed["NASDAQCOM"] / observed["NASDAQCOM"].shift())
    returns = observed["Log return"].dropna()
    return frame, design, observed, returns, float(returns.std(ddof=1) * np.sqrt(252))


def cashflow_components(prices, positions, strike, cost_rate, premium=0.0):
    zero = torch.zeros_like(positions[:, :1])
    trades = torch.diff(torch.cat([zero, positions, zero], dim=1), dim=1)
    gains = (positions * torch.diff(prices, dim=1)).sum(dim=1)
    turnover = (prices * trades.abs()).sum(dim=1)
    costs = cost_rate * turnover
    payoff = torch.relu(prices[:, -1] - strike)
    pnl = premium + gains - costs - payoff
    return {"trades": trades, "gains": gains, "turnover": turnover, "costs": costs, "payoff": payoff, "pnl": pnl}


def metric_record(name, prices, positions, strike, cost_rate, risk_aversion):
    components = cashflow_components(prices, positions, strike, cost_rate)
    pnl = components["pnl"].detach()
    cutoff = torch.quantile(pnl, 0.05)
    return {
        "Strategy": name,
        "Entropic risk": entropic_risk(pnl, risk_aversion).item(),
        "Mean cash flow": pnl.mean().item(),
        "Cash flow standard deviation": pnl.std(unbiased=True).item(),
        "Fifth percentile": cutoff.item(),
        "Mean loss in worst five percent": -pnl[pnl <= cutoff].mean().item(),
        "Mean turnover": components["turnover"].mean().item(),
        "Mean trading cost": components["costs"].mean().item(),
        "Mean payoff": components["payoff"].mean().item(),
    }

full_public, public_design, public_observed, public_returns, sigma = public_volatility()
design = {
    "spot": 1.0,
    "strike": 1.0,
    "maturity_days": 30,
    "maturity": 30 / 365,
    "volatility": sigma,
    "drift": 0.0,
    "steps": 20,
    "cost_rate": 0.002,
    "risk_aversion": 3.0,
    "hidden_size": 16,
    "learning_rate": 0.003,
    "training_paths": 2048,
    "evaluation_paths": 8192,
    "iterations": 200,
}
training_seeds = [3301, 3302, 3303]
evaluation_seed = 33999
evaluation_prices = gbm_hedging_paths(spot=design["spot"], maturity=design["maturity"], volatility=design["volatility"], drift=design["drift"], steps=design["steps"], paths=design["evaluation_paths"], seed=evaluation_seed)

strategies = {}
for seed in training_seeds:
    model = DeepHedge(design["hidden_size"], seed=seed * 10 + 2)
    model.load_state_dict(torch.load(ARTIFACTS / f"deep_hedge_seed_{seed}.pt", weights_only=True))
    strategies[f"Learned seed {seed}"] = model(evaluation_prices, strike=design["strike"], maturity=design["maturity"]).detach()
strategies["Black-Scholes delta"] = black_scholes_call_delta(evaluation_prices, strike=design["strike"], maturity=design["maturity"], volatility=design["volatility"])
strategies["Zero hedge"] = torch.zeros_like(strategies["Black-Scholes delta"])

metrics = pd.DataFrame([metric_record(name, evaluation_prices, positions, design["strike"], design["cost_rate"], design["risk_aversion"]) for name, positions in strategies.items()])
save_split_table(metrics, [("Risk and cash flow distribution", ["Strategy", "Entropic risk", "Mean cash flow", "Cash flow standard deviation", "Fifth percentile"]), ("Tail, turnover, cost, and payoff", ["Strategy", "Mean loss in worst five percent", "Mean turnover", "Mean trading cost", "Mean payoff"])], "deep_hedging_comparator_metrics", "Independent outcomes for all learned policies and both comparators.", "tab:chapter33-comparator-metrics")

ranking = metrics[["Strategy", "Entropic risk"]].sort_values("Entropic risk").reset_index(drop=True)
ranking.insert(0, "Rank in this sample", np.arange(1, len(ranking) + 1))
ranking["Difference from delta"] = ranking["Entropic risk"] - float(metrics.loc[metrics["Strategy"] == "Black-Scholes delta", "Entropic risk"].iloc[0])
save_table(ranking, "deep_hedging_comparator_ranking", "Risk ordering in this sample without policy selection.", "tab:chapter33-comparator-ranking", fit=True)

recalculation_rows = []
pnl_frame = pd.DataFrame()
for name, positions in strategies.items():
    components = cashflow_components(evaluation_prices, positions, design["strike"], design["cost_rate"])
    released = hedging_pnl(evaluation_prices, positions, strike=design["strike"], cost_rate=design["cost_rate"])
    recalculation_rows.append({"Strategy": name, "Maximum cash flow difference": torch.max(torch.abs(released - components["pnl"])).item(), "Pass": bool(torch.allclose(released, components["pnl"], atol=1e-13, rtol=0))})
    pnl_frame[name] = released.detach().numpy()
recalculation = pd.DataFrame(recalculation_rows)
save_table(recalculation, "deep_hedging_comparator_recalculation", "Independent cash flow checks for every reported strategy.", "tab:chapter33-comparator-recalculation", fit=True)

evaluation_contract = pd.DataFrame([
    {"Item": "Evaluation paths", "Value": design["evaluation_paths"]},
    {"Item": "Evaluation seed", "Value": evaluation_seed},
    {"Item": "Same paths for every strategy", "Value": "Yes"},
    {"Item": "Evaluation used to choose a training seed", "Value": "No"},
    {"Item": "Evaluation used to choose stopping time", "Value": "No"},
    {"Item": "Claim of global optimality", "Value": "None"},
])
save_table(evaluation_contract, "deep_hedging_evaluation_contract", "Rules for the independent strategy comparison.", "tab:chapter33-evaluation-contract", fit=True)

risk_figure = metrics[["Strategy", "Entropic risk"]].copy()
risk_figure.to_csv(FIGURE_DATA / "deep_hedging_comparator_risk.csv", index=False)
figure, axis = plt.subplots(figsize=(9.2, 5.2), constrained_layout=True)
colors = [PALETTE["cyan"], PALETTE["amber"], PALETTE["green"], PALETTE["violet"], PALETTE["rose"]]
axis.bar(risk_figure["Strategy"], risk_figure["Entropic risk"], color=colors)
axis.tick_params(axis="x", rotation=24)
axis.set(title="Entropic Risk on Common Independent Paths", xlabel="Strategy", ylabel="Entropic risk")
finish_axis(axis, grid=False)
save_figure(figure, "deep_hedging_comparator_risk")

tail_cost = metrics[["Strategy", "Mean trading cost", "Mean loss in worst five percent", "Entropic risk"]].copy()
tail_cost.to_csv(FIGURE_DATA / "deep_hedging_tail_cost_comparison.csv", index=False)
figure, axis = plt.subplots(figsize=(9.0, 5.2), constrained_layout=True)
for index, row in tail_cost.iterrows():
    axis.scatter(row["Mean trading cost"], row["Mean loss in worst five percent"], s=65, color=colors[index], label=row["Strategy"])
axis.set(title="Trading Cost and Adverse Cash Flow Tail", xlabel="Mean trading cost", ylabel="Mean loss in worst five percent")
axis.legend(fontsize=8)
finish_axis(axis)
save_figure(figure, "deep_hedging_tail_cost_comparison")

pnl_frame.to_csv(FIGURE_DATA / "deep_hedging_comparator_cashflows.csv", index=False)
figure, axis = plt.subplots(figsize=(9.0, 5.2), constrained_layout=True)
for name, color in zip(pnl_frame.columns, colors):
    axis.hist(pnl_frame[name], bins=70, density=True, histtype="step", linewidth=1.3, label=name, color=color)
axis.set(title="Cash Flow Distributions on Common Independent Paths", xlabel="Terminal cash flow", ylabel="Density")
axis.legend(fontsize=8)
finish_axis(axis)
save_figure(figure, "deep_hedging_comparator_cashflows")

mean_holdings = pd.DataFrame({"Trading date": np.arange(design["steps"])})
for name, positions in strategies.items():
    mean_holdings[name] = positions.mean(dim=0).numpy()
mean_holdings.to_csv(FIGURE_DATA / "deep_hedging_comparator_holdings.csv", index=False)
figure, axis = plt.subplots(figsize=(9.0, 5.2), constrained_layout=True)
for name, color in zip(mean_holdings.columns[1:], colors):
    axis.plot(mean_holdings["Trading date"], mean_holdings[name], linewidth=1.4, label=name, color=color)
axis.set(title="Mean Holding through the Trading Horizon", xlabel="Trading date", ylabel="Mean holding")
axis.legend(fontsize=8)
finish_axis(axis)
save_figure(figure, "deep_hedging_comparator_holdings")

,Strategy,Entropic risk,Mean cash flow,Cash flow standard deviation,Fifth percentile,Mean loss in worst five percent,Mean turnover,Mean trading cost,Mean payoff
0,Learned seed 3301,0.031404,-0.030042,0.029301,-0.094589,0.11859,0.530783,0.001062,0.028887
1,Learned seed 3302,0.031523,-0.030631,0.023963,-0.078803,0.097747,0.800167,0.0016,0.028887
2,Learned seed 3303,0.031436,-0.030395,0.025773,-0.084628,0.105965,0.692083,0.001384,0.028887
3,Black-Scholes delta,0.033871,-0.033818,0.00593,-0.044228,0.048505,2.400615,0.004801,0.028887
4,Zero hedge,0.032104,-0.028887,0.044459,-0.127133,0.159703,0,0,0.028887


,Rank in this sample,Strategy,Entropic risk,Difference from delta
0,1,Learned seed 3301,0.031404,-0.002467
1,2,Learned seed 3303,0.031436,-0.002436
2,3,Learned seed 3302,0.031523,-0.002349
3,4,Zero hedge,0.032104,-0.001767
4,5,Black-Scholes delta,0.033871,0


,Strategy,Maximum cash flow difference,Pass
0,Learned seed 3301,0,Pass
1,Learned seed 3302,0,Pass
2,Learned seed 3303,0,Pass
3,Black-Scholes delta,0,Pass
4,Zero hedge,0,Pass


,Item,Value
0,Evaluation paths,"8,192"
1,Evaluation seed,"33,999"
2,Same paths for every strategy,Yes
3,Evaluation used to choose a training seed,No
4,Evaluation used to choose stopping time,No
5,Claim of global optimality,None


<Figure size 920x520 with 1 Axes>

<Figure size 900x520 with 1 Axes>

<Figure size 900x520 with 1 Axes>

<Figure size 900x520 with 1 Axes>

## Sensitivity with One Change at a Time

For each sensitivity run, define the comparator gap as

$$
d=\widehat\rho_{\mathrm{learned}}-\widehat\rho_{\Delta}.
$$

Here, $\widehat\rho_{\mathrm{learned}}$ is the learned policy's entropic risk in this sample and $\widehat\rho_{\Delta}$ is the Black-Scholes delta risk on the same evaluation paths. A negative $d$ favors the learned policy in that declared run; a positive $d$ favors the delta comparator. This difference does not establish a general ranking.

### Testing hedge sensitivity across design choices

In [2]:
sensitivity_contract = {
    "Training paths per run": 384,
    "Evaluation paths per run": 2048,
    "Training iterations per run": 50,
    "Hidden units per layer": 16,
    "Learning rate": 0.003,
    "Training path seed": 33701,
    "Policy seed": 33702,
    "Evaluation seed": 33703,
}
save_table(pd.DataFrame([{"Item": key, "Value": value} for key, value in sensitivity_contract.items()]), "deep_hedging_sensitivity_contract", "Fixed contract for every sensitivity run with one change at a time.", "tab:chapter33-sensitivity-contract", fit=True)

def sensitivity_run(factor, value, cost_rate, risk_aversion, steps, maturity_days):
    maturity = maturity_days / 365
    training_prices = gbm_hedging_paths(spot=design["spot"], maturity=maturity, volatility=design["volatility"], drift=design["drift"], steps=steps, paths=sensitivity_contract["Training paths per run"], seed=sensitivity_contract["Training path seed"])
    evaluation = gbm_hedging_paths(spot=design["spot"], maturity=maturity, volatility=design["volatility"], drift=design["drift"], steps=steps, paths=sensitivity_contract["Evaluation paths per run"], seed=sensitivity_contract["Evaluation seed"])
    fitted = DeepHedge(sensitivity_contract["Hidden units per layer"], seed=sensitivity_contract["Policy seed"])
    started = time.perf_counter()
    history = train_deep_hedge(fitted, training_prices, strike=design["strike"], maturity=maturity, cost_rate=cost_rate, risk_aversion=risk_aversion, iterations=sensitivity_contract["Training iterations per run"], learning_rate=sensitivity_contract["Learning rate"])
    elapsed = time.perf_counter() - started
    fitted_positions = fitted(evaluation, strike=design["strike"], maturity=maturity).detach()
    delta_positions = black_scholes_call_delta(evaluation, strike=design["strike"], maturity=maturity, volatility=design["volatility"])
    zero_positions = torch.zeros_like(delta_positions)
    learned_risk = entropic_risk(hedging_pnl(evaluation, fitted_positions, strike=design["strike"], cost_rate=cost_rate), risk_aversion).item()
    delta_risk = entropic_risk(hedging_pnl(evaluation, delta_positions, strike=design["strike"], cost_rate=cost_rate), risk_aversion).item()
    zero_risk = entropic_risk(hedging_pnl(evaluation, zero_positions, strike=design["strike"], cost_rate=cost_rate), risk_aversion).item()
    return {
        "Factor": factor,
        "Value": value,
        "Cost rate": cost_rate,
        "Risk aversion": risk_aversion,
        "Trading steps": steps,
        "Maturity days": maturity_days,
        "Initial training risk": history[0],
        "Final training risk": history[-1],
        "Learned risk": learned_risk,
        "Delta risk": delta_risk,
        "Zero hedge risk": zero_risk,
        "Learned minus delta": learned_risk - delta_risk,
        "Training seconds": elapsed,
        "Finite": bool(np.isfinite(history).all() and np.isfinite([learned_risk, delta_risk, zero_risk]).all()),
    }

sensitivity_rows = []
for value in [0.0, 0.001, 0.002, 0.005]:
    sensitivity_rows.append(sensitivity_run("Trading cost", value, value, design["risk_aversion"], design["steps"], design["maturity_days"]))
for value in [1.0, 3.0, 10.0]:
    sensitivity_rows.append(sensitivity_run("Risk aversion", value, design["cost_rate"], value, design["steps"], design["maturity_days"]))
for value in [10, 20, 40]:
    sensitivity_rows.append(sensitivity_run("Trading steps", value, design["cost_rate"], design["risk_aversion"], value, design["maturity_days"]))
for value in [30, 60, 90, 180, 360]:
    sensitivity_rows.append(sensitivity_run("Maturity days", value, design["cost_rate"], design["risk_aversion"], design["steps"], value))
sensitivity = pd.DataFrame(sensitivity_rows)
assert len(sensitivity) == 15 and sensitivity["Finite"].all()
sensitivity.to_csv(ARTIFACTS / "deep_hedging_sensitivity_raw.csv", index=False)

table_specs = [
    ("Trading cost", "deep_hedging_cost_sensitivity", "Sensitivity to trading cost under the fixed smaller training contract.", "tab:chapter33-cost-sensitivity"),
    ("Risk aversion", "deep_hedging_risk_sensitivity", "Sensitivity to risk aversion under the fixed smaller training contract.", "tab:chapter33-risk-sensitivity"),
    ("Trading steps", "deep_hedging_step_sensitivity", "Sensitivity to the number of trading steps under the fixed smaller training contract.", "tab:chapter33-step-sensitivity"),
    ("Maturity days", "deep_hedging_maturity_sensitivity", "Maturity sensitivity under the fixed smaller training contract.", "tab:chapter33-maturity-sensitivity"),
]
for factor, stem, caption, label in table_specs:
    selected = sensitivity.loc[sensitivity["Factor"] == factor, ["Value", "Learned risk", "Delta risk", "Zero hedge risk", "Learned minus delta", "Training seconds", "Finite"]].copy()
    save_table(selected, stem, caption, label, fit=True)
    selected.to_csv(FIGURE_DATA / f"{stem}.csv", index=False)
    figure, axis = plt.subplots(figsize=(9.0, 5.2), constrained_layout=True)
    axis.plot(selected["Value"], selected["Learned risk"], marker="o", color=PALETTE["cyan"], label="Learned policy")
    axis.plot(selected["Value"], selected["Delta risk"], marker="s", color=PALETTE["amber"], label="Black-Scholes delta")
    axis.plot(selected["Value"], selected["Zero hedge risk"], marker="^", color=PALETTE["rose"], label="Zero hedge")
    axis.set(title=caption.rstrip("."), xlabel=factor, ylabel="Entropic risk")
    axis.legend()
    finish_axis(axis)
    save_figure(figure, stem)

,Item,Value
0,Training paths per run,384
1,Evaluation paths per run,"2,048"
2,Training iterations per run,50
3,Hidden units per layer,16
4,Learning rate,0.003
5,Training path seed,"33,701"
6,Policy seed,"33,702"
7,Evaluation seed,"33,703"


,Value,Learned risk,Delta risk,Zero hedge risk,Learned minus delta,Training seconds,Finite
0,0,0.029066,0.028877,0.032951,0.000189,1.270898,Pass
1,0.001,0.030378,0.03128,0.032951,-0.000902,0.197339,Pass
2,0.002,0.031379,0.033685,0.032951,-0.002306,0.19627,Pass
3,0.005,0.032903,0.040907,0.032951,-0.008004,0.202357,Pass


<Figure size 900x520 with 1 Axes>

,Value,Learned risk,Delta risk,Zero hedge risk,Learned minus delta,Training seconds,Finite
4,1,0.030717,0.033652,0.030712,-0.002935,0.192827,Pass
5,3,0.031379,0.033685,0.032951,-0.002306,0.196619,Pass
6,10,0.03328,0.033803,0.043348,-0.000523,0.194286,Pass


<Figure size 900x520 with 1 Axes>

,Value,Learned risk,Delta risk,Zero hedge risk,Learned minus delta,Training seconds,Finite
7,10,0.034344,0.03294,0.0347,0.001403,0.119716,Pass
8,20,0.031379,0.033685,0.032951,-0.002306,0.193997,Pass
9,40,0.031613,0.035055,0.032518,-0.003442,0.345378,Pass


<Figure size 900x520 with 1 Axes>

,Value,Learned risk,Delta risk,Zero hedge risk,Learned minus delta,Training seconds,Finite
10,30,0.031379,0.033685,0.032951,-0.002306,0.195048,Pass
11,60,0.044053,0.04568,0.049087,-0.001627,0.197426,Pass
12,90,0.053902,0.054887,0.062872,-0.000985,0.194137,Pass
13,180,0.076398,0.075658,0.100645,0.00074,0.191513,Pass
14,360,0.108622,0.105012,0.190949,0.00361,0.20816,Pass


<Figure size 900x520 with 1 Axes>

## Checking seed variation within the declared scope

The retained seed spread is

$$
R_{\mathrm{seed}}=\max_j\widehat\rho_j-\min_j\widehat\rho_j.
$$

Here, $\widehat\rho_j$ is the independent evaluation risk for declared training seed $j$. The range $R_{\mathrm{seed}}$ describes fitting variation in this fixed experiment. It is not uncertainty about all possible model choices or future markets.

In [3]:
primary_metrics = pd.read_csv(ARTIFACTS / "primary_evaluation_metrics.csv")
seed_sensitivity = primary_metrics[["Strategy", "Entropic risk", "Mean cash flow", "Cash flow standard deviation", "Mean trading cost"]].copy()
seed_sensitivity["Seed"] = seed_sensitivity["Strategy"].str.extract(r"(\d+)").astype(int)
seed_sensitivity = seed_sensitivity[["Seed", "Entropic risk", "Mean cash flow", "Cash flow standard deviation", "Mean trading cost"]]
save_table(seed_sensitivity, "deep_hedging_seed_sensitivity", "Independent results for every primary training seed.", "tab:chapter33-seed-sensitivity", fit=True)

seed_range = pd.DataFrame([
    {"Quantity": "Smallest entropic risk", "Value": seed_sensitivity["Entropic risk"].min()},
    {"Quantity": "Largest entropic risk", "Value": seed_sensitivity["Entropic risk"].max()},
    {"Quantity": "Seed risk range", "Value": seed_sensitivity["Entropic risk"].max() - seed_sensitivity["Entropic risk"].min()},
    {"Quantity": "Declared seeds retained", "Value": len(seed_sensitivity)},
])
save_table(seed_range, "deep_hedging_seed_range", "Observed spread across all declared primary seeds.", "tab:chapter33-seed-range", fit=True)

timing = sensitivity.groupby("Factor", as_index=False).agg(Runs=("Value", "count"), Mean_seconds=("Training seconds", "mean"), Maximum_seconds=("Training seconds", "max"), All_finite=("Finite", "all")).rename(columns={"Mean_seconds": "Mean training seconds", "Maximum_seconds": "Maximum training seconds", "All_finite": "All runs finite"})
save_table(timing, "deep_hedging_sensitivity_timing", "Observed computer time for the bounded sensitivity groups.", "tab:chapter33-sensitivity-timing", fit=True)

directions = sensitivity.groupby("Factor", as_index=False).agg(Runs=("Value", "count"), Learned_lower=("Learned minus delta", lambda values: int((values < 0).sum())), Delta_lower=("Learned minus delta", lambda values: int((values > 0).sum())), Ties=("Learned minus delta", lambda values: int((np.abs(values) <= 1e-12).sum())))
directions = directions.rename(columns={"Learned_lower": "Learned risk lower", "Delta_lower": "Delta risk lower"})
save_table(directions, "deep_hedging_sensitivity_directions", "Comparison directions across the declared sensitivity runs.", "tab:chapter33-sensitivity-directions", fit=True)

scope = pd.DataFrame([
    {"Question": "Does the chapter prove a global optimum?", "Answer": "No", "Boundary": "Fixed finite training only"},
    {"Question": "Does it establish market performance?", "Answer": "No", "Boundary": "Generated paths only"},
    {"Question": "Does it claim superiority over delta?", "Answer": "No", "Boundary": "All observed rankings are reported"},
    {"Question": "Is the path law fitted to option prices?", "Answer": "No", "Boundary": "Public volatility scale only"},
    {"Question": "Are evaluation paths used for selection?", "Answer": "No", "Boundary": "Independent final evaluation"},
    {"Question": "Are private preliminary architectures copied?", "Answer": "No", "Boundary": "Released RiskLabAI method only"},
])
save_table(scope, "deep_hedging_scope_boundaries", "Interpretation boundaries for the generated hedging study.", "tab:chapter33-scope-boundaries", fit=True)

seed_sensitivity.to_csv(FIGURE_DATA / "deep_hedging_seed_sensitivity.csv", index=False)
figure, axis = plt.subplots(figsize=(8.5, 4.8), constrained_layout=True)
axis.plot(seed_sensitivity["Seed"].astype(str), seed_sensitivity["Entropic risk"], marker="o", color=PALETTE["green"], linewidth=1.7)
axis.set(title="Independent Risk for Every Declared Training Seed", xlabel="Training seed", ylabel="Entropic risk")
finish_axis(axis)
save_figure(figure, "deep_hedging_seed_sensitivity")

timing_figure = sensitivity[["Factor", "Value", "Training seconds"]].copy()
timing_figure["Run"] = timing_figure["Factor"] + ": " + timing_figure["Value"].map(rendered_value)
timing_figure.to_csv(FIGURE_DATA / "deep_hedging_sensitivity_training_time.csv", index=False)
figure, axis = plt.subplots(figsize=(10.0, 5.4), constrained_layout=True)
axis.bar(np.arange(len(timing_figure)), timing_figure["Training seconds"], color=PALETTE["violet"])
axis.set_xticks(np.arange(len(timing_figure)))
axis.set_xticklabels(timing_figure["Run"], rotation=62, ha="right", fontsize=8)
axis.set(title="Observed Training Time for Every Sensitivity Run", xlabel="Declared run", ylabel="Seconds on this computer")
finish_axis(axis, grid=False)
save_figure(figure, "deep_hedging_sensitivity_training_time")

,Seed,Entropic risk,Mean cash flow,Cash flow standard deviation,Mean trading cost
0,"3,301",0.031404,-0.030042,0.029301,0.001062
1,"3,302",0.031523,-0.030631,0.023963,0.0016
2,"3,303",0.031436,-0.030395,0.025773,0.001384


,Quantity,Value
0,Smallest entropic risk,0.031404
1,Largest entropic risk,0.031523
2,Seed risk range,0.000119
3,Declared seeds retained,3


,Factor,Runs,Mean training seconds,Maximum training seconds,All runs finite
0,Maturity days,5,0.197257,0.20816,Pass
1,Risk aversion,3,0.194577,0.196619,Pass
2,Trading cost,4,0.466716,1.270898,Pass
3,Trading steps,3,0.219697,0.345378,Pass


,Factor,Runs,Learned risk lower,Delta risk lower,Ties
0,Maturity days,5,3,2,0
1,Risk aversion,3,3,0,0
2,Trading cost,4,3,1,0
3,Trading steps,3,2,1,0


,Question,Answer,Boundary
0,Does the chapter prove a global optimum?,No,Fixed finite training only
1,Does it establish market performance?,No,Generated paths only
2,Does it claim superiority over delta?,No,All observed rankings are reported
3,Is the path law fitted to option prices?,No,Public volatility scale only
4,Are evaluation paths used for selection?,No,Independent final evaluation
5,Are private preliminary architectures copied?,No,Released RiskLabAI method only


<Figure size 850x480 with 1 Axes>

<Figure size 1000x540 with 1 Axes>